# B3 · Attention et Transformers : un mini-GPT from scratch — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b3_transformers/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b3_transformers/03_notebook.ipynb`, créée par `python tools/start_chapter.py B3`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B3.Q1 à B3.Q12, B3.R1 à B3.R3, B3.1 à B3.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B3.12 à B3.17 | ta librairie NumPy : masques et attention, carte d'attention à prédire, têtes, positions sinusoïdales et RoPE, bloc Transformer complet validé contre PyTorch, trois masques qui mentent (*prochaine session de génération*) | | | |
| B | B3.18 à B3.30 | le mini-GPT en PyTorch : module d'attention, modèle, entraînement sur Holmes, courbes, Holmes amorcé en français, KV cache, cartes d'attention, ablations, BPE contre caractères, contexte long, ViT, test de causalité, défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B3")  # your own library

## Objectifs et rappel express

- Calculer à la main une attention par produit scalaire, avec et sans masque causal, et justifier la division par $\sqrt{d_k}$.
- Implémenter en NumPy les masques, l'attention multi-têtes, les encodages de position (sinusoïdal, RoPE) et un bloc Transformer pré-LN, et les valider contre PyTorch.
- Écrire et entraîner en PyTorch un mini-GPT sur Holmes, le faire générer avec un KV cache, et comparer sa perplexité à celle du LSTM du ch. 22.
- Estimer les paramètres, les FLOPs et la mémoire d'un Transformer, et mesurer le coût d'un long contexte.

**Rappel express.** $\operatorname{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \operatorname{softmax}\big(\mathbf{Q}\mathbf{K}^\top / \sqrt{d_k} + \mathbf{M}\big)\mathbf{V}$, la softmax sur les clés, $M_{ij} = -\infty$ là où le masque vaut `False`. Masque causal : la requête $i$ voit les clés $j \le i + S - L$. Multi-têtes : $(N, T, d) \to (N, H, T, d_h)$, $4d^2 + 4d$ paramètres. Encodage sinusoïdal : $PE_{p, 2i} = \sin(p\,\omega_i)$, $PE_{p, 2i+1} = \cos(p\,\omega_i)$, $\omega_i = 10000^{-2i/d}$. RoPE : $\langle R_m \mathbf{q}, R_n \mathbf{k} \rangle = \langle \mathbf{q}, R_{n-m} \mathbf{k} \rangle$. Bloc pré-LN : $\mathbf{h} = \mathbf{x} + \operatorname{MHA}(\operatorname{LN}_1(\mathbf{x}))$, $\mathbf{y} = \mathbf{h} + \operatorname{MLP}(\operatorname{LN}_2(\mathbf{h}))$, environ $12d^2$ paramètres. Entraînement : environ $6ND$ FLOPs ; KV cache : $2 \times L \times T \times d \times$ octets par valeur.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B3.1 à B3.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; une forme de tableau : entre guillemets (`"(2, 3)"`) ; un vecteur ou une ligne de matrice : une liste (`[0.5, -1, 3]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B3.6 et B3.7, l'estimation 🧮 B3.8, la réflexion (🗣️ B3.9, ⚖️ B3.10, 📄 B3.11) et l'entretien se corrigent avec `05_solutions.md`.

**B3.Q1 — Le goulot de l'encodeur-décodeur récurrent : pourquoi un seul vecteur ne suffit pas**

In [ ]:
# B3.Q1: the values you found on paper
answer_B3_Q1a = ...  # a) the letter of your choice, e.g. "B"
answer_B3_Q1b = ...  # b) True or False
answer_B3_Q1c = ...  # c) a whole number
answer_B3_Q1d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q1a, answer_B3_Q1b, answer_B3_Q1c, answer_B3_Q1d, answer_B3_Q1e]):
    wb.check(f"B3.Q1{letter}", answer)

**B3.Q2 — Requête, clé, valeur : un dictionnaire où l'on cherche « à peu près »**

In [ ]:
# B3.Q2: the values you found on paper
answer_B3_Q2a = ...  # a) the letter of your choice, e.g. "B"
answer_B3_Q2b = ...  # b) True or False
answer_B3_Q2c = ...  # c) a number, 2 decimals
answer_B3_Q2d = ...  # d) a number, 1 decimal
answer_B3_Q2e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B3_Q2a, answer_B3_Q2b, answer_B3_Q2c, answer_B3_Q2d, answer_B3_Q2e]):
    wb.check(f"B3.Q2{letter}", answer)

**B3.Q3 — Pourquoi diviser les scores par √d_k ?**

In [ ]:
# B3.Q3: the values you found on paper
answer_B3_Q3a = ...  # a) a whole number
answer_B3_Q3b = ...  # b) True or False
answer_B3_Q3c = ...  # c) the letter of your choice, e.g. "B"
answer_B3_Q3d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q3a, answer_B3_Q3b, answer_B3_Q3c, answer_B3_Q3d, answer_B3_Q3e]):
    wb.check(f"B3.Q3{letter}", answer)

**B3.Q4 — Masque causal : qui a le droit de regarder qui ?**

In [ ]:
# B3.Q4: the values you found on paper
answer_B3_Q4a = ...  # a) a whole number
answer_B3_Q4b = ...  # b) the letter of your choice, e.g. "B"
answer_B3_Q4c = ...  # c) True or False
answer_B3_Q4d = ...  # d) a whole number
answer_B3_Q4e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B3_Q4a, answer_B3_Q4b, answer_B3_Q4c, answer_B3_Q4d, answer_B3_Q4e]):
    wb.check(f"B3.Q4{letter}", answer)

**B3.Q5 — Auto-attention ou attention croisée : d'où viennent Q, K et V ?**

In [ ]:
# B3.Q5: the values you found on paper
answer_B3_Q5a = ...  # a) the letter of your choice, e.g. "B"
answer_B3_Q5b = ...  # b) a shape as a string, e.g. "(2, 3)"
answer_B3_Q5c = ...  # c) True or False
answer_B3_Q5d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q5a, answer_B3_Q5b, answer_B3_Q5c, answer_B3_Q5d, answer_B3_Q5e]):
    wb.check(f"B3.Q5{letter}", answer)

**B3.Q6 — Plusieurs têtes pour le prix d'une : que gagne-t-on ?**

In [ ]:
# B3.Q6: the values you found on paper
answer_B3_Q6a = ...  # a) a whole number
answer_B3_Q6b = ...  # b) True or False
answer_B3_Q6c = ...  # c) a whole number
answer_B3_Q6d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q6a, answer_B3_Q6b, answer_B3_Q6c, answer_B3_Q6d, answer_B3_Q6e]):
    wb.check(f"B3.Q6{letter}", answer)

**B3.Q7 — Sans encodage de position, l'attention voit-elle l'ordre des mots ?**

In [ ]:
# B3.Q7: the values you found on paper
answer_B3_Q7a = ...  # a) True or False
answer_B3_Q7b = ...  # b) the letter of your choice, e.g. "B"
answer_B3_Q7c = ...  # c) the letter of your choice, e.g. "B"
answer_B3_Q7d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_Q7e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B3_Q7a, answer_B3_Q7b, answer_B3_Q7c, answer_B3_Q7d, answer_B3_Q7e]):
    wb.check(f"B3.Q7{letter}", answer)

**B3.Q8 — Résiduel, pré-LN ou post-LN : ce qui rend un Transformer profond entraînable**

In [ ]:
# B3.Q8: the values you found on paper
answer_B3_Q8a = ...  # a) the letter of your choice, e.g. "B"
answer_B3_Q8b = ...  # b) True or False
answer_B3_Q8c = ...  # c) the letter of your choice, e.g. "B"
answer_B3_Q8d = ...  # d) a whole number
answer_B3_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q8a, answer_B3_Q8b, answer_B3_Q8c, answer_B3_Q8d, answer_B3_Q8e]):
    wb.check(f"B3.Q8{letter}", answer)

**B3.Q9 — BERT, GPT, T5 : quel masque, quel objectif, quel usage ?**

In [ ]:
# B3.Q9: the values you found on paper
answer_B3_Q9a = ...  # a) the letter of your choice, e.g. "B"
answer_B3_Q9b = ...  # b) a whole number
answer_B3_Q9c = ...  # c) a whole number
answer_B3_Q9d = ...  # d) True or False
answer_B3_Q9e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B3_Q9a, answer_B3_Q9b, answer_B3_Q9c, answer_B3_Q9d, answer_B3_Q9e]):
    wb.check(f"B3.Q9{letter}", answer)

**B3.Q10 — Décalage d'un token : ce que chaque position apprend à prédire**

In [ ]:
# B3.Q10: the values you found on paper
answer_B3_Q10a = ...  # a) the letter of your choice, e.g. "B"
answer_B3_Q10b = ...  # b) a whole number
answer_B3_Q10c = ...  # c) True or False
answer_B3_Q10d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_Q10e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_B3_Q10a, answer_B3_Q10b, answer_B3_Q10c, answer_B3_Q10d, answer_B3_Q10e]):
    wb.check(f"B3.Q10{letter}", answer)

**B3.Q11 — KV cache : ce que l'on garde et pourquoi la génération s'accélère**

In [ ]:
# B3.Q11: the values you found on paper
answer_B3_Q11a = ...  # a) the letter of your choice, e.g. "B"
answer_B3_Q11b = ...  # b) a whole number
answer_B3_Q11c = ...  # c) True or False
answer_B3_Q11d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_Q11e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B3_Q11a, answer_B3_Q11b, answer_B3_Q11c, answer_B3_Q11d, answer_B3_Q11e]):
    wb.check(f"B3.Q11{letter}", answer)

**B3.Q12 — Coût quadratique du contexte et ce que change FlashAttention**

In [ ]:
# B3.Q12: the values you found on paper
answer_B3_Q12a = ...  # a) a whole number
answer_B3_Q12b = ...  # b) True or False
answer_B3_Q12c = ...  # c) the letter of your choice, e.g. "B"
answer_B3_Q12d = ...  # d) a whole number
answer_B3_Q12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_Q12a, answer_B3_Q12b, answer_B3_Q12c, answer_B3_Q12d, answer_B3_Q12e]):
    wb.check(f"B3.Q12{letter}", answer)

**B3.R1 — De quoi dépend le nombre de tokens d'un texte ?**

In [ ]:
# B3.R1: the values you found on paper
answer_B3_R1a = ...  # a) True or False
answer_B3_R1b = ...  # b) a whole number
answer_B3_R1c = ...  # c) the letter of your choice, e.g. "B"
answer_B3_R1d = ...  # d) True or False
answer_B3_R1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B3_R1a, answer_B3_R1b, answer_B3_R1c, answer_B3_R1d, answer_B3_R1e]):
    wb.check(f"B3.R1{letter}", answer)

**B3.R2 — Pourquoi le gradient s'évanouit dans un RNN déroulé**

In [ ]:
# B3.R2: the values you found on paper
answer_B3_R2a = ...  # a) a number, 4 decimals
answer_B3_R2b = ...  # b) True or False
answer_B3_R2c = ...  # c) the letter of your choice, e.g. "B"
answer_B3_R2d = ...  # d) the letter of your choice, e.g. "B"
answer_B3_R2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B3_R2a, answer_B3_R2b, answer_B3_R2c, answer_B3_R2d, answer_B3_R2e]):
    wb.check(f"B3.R2{letter}", answer)

**B3.R3 — Layer norm et connexion résiduelle, à quoi sert chacune ?**

In [ ]:
# B3.R3: the values you found on paper
answer_B3_R3a = ...  # a) a number, 3 decimals
answer_B3_R3b = ...  # b) True or False
answer_B3_R3c = ...  # c) the letter of your choice, e.g. "B"
answer_B3_R3d = ...  # d) True or False
answer_B3_R3e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B3_R3a, answer_B3_R3b, answer_B3_R3c, answer_B3_R3d, answer_B3_R3e]):
    wb.check(f"B3.R3{letter}", answer)

**Ex B3.1 — L'attention à la main : trois tokens, d_k = 2, scores, softmax et sortie**

In [ ]:
# Ex B3.1: the values you found on paper
answer_B3_1a = ...  # a) a list of numbers, e.g. [0.5, -1, 3]
answer_B3_1b = ...  # b) a number, 3 decimals
answer_B3_1c = ...  # c) a number, 3 decimals
answer_B3_1d = ...  # d) a list of numbers, 3 decimals each
answer_B3_1e = ...  # e) a number, 3 decimals
answer_B3_1f = ...  # f) True or False
answer_B3_1g = ...  # g) a number, 3 decimals

for letter, answer in zip("abcdefg", [answer_B3_1a, answer_B3_1b, answer_B3_1c, answer_B3_1d, answer_B3_1e, answer_B3_1f, answer_B3_1g]):
    wb.check(f"B3.1{letter}", answer)

**Ex B3.2 — Le même calcul avec un masque causal**

In [ ]:
# Ex B3.2: the values you found on paper
answer_B3_2a = ...  # a) a whole number
answer_B3_2b = ...  # b) a list of numbers, 3 decimals each
answer_B3_2c = ...  # c) a list of numbers, 3 decimals each
answer_B3_2d = ...  # d) a list of numbers, 3 decimals each
answer_B3_2e = ...  # e) True or False
answer_B3_2f = ...  # f) a whole number
answer_B3_2g = ...  # g) a whole number
answer_B3_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B3_2a, answer_B3_2b, answer_B3_2c, answer_B3_2d, answer_B3_2e, answer_B3_2f, answer_B3_2g, answer_B3_2h]):
    wb.check(f"B3.2{letter}", answer)

**Ex B3.3 — Suivre les formes dans l'attention multi-têtes et compter ses paramètres**

In [ ]:
# Ex B3.3: the values you found on paper
answer_B3_3a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B3_3b = ...  # b) a shape as a string, e.g. "(2, 3)"
answer_B3_3c = ...  # c) a shape as a string, e.g. "(2, 3)"
answer_B3_3d = ...  # d) a whole number
answer_B3_3e = ...  # e) a whole number
answer_B3_3f = ...  # f) True or False
answer_B3_3g = ...  # g) a shape as a string, e.g. "(2, 3)"
answer_B3_3h = ...  # h) a whole number
answer_B3_3i = ...  # i) a shape as a string, e.g. "(2, 3)"

for letter, answer in zip("abcdefghi", [answer_B3_3a, answer_B3_3b, answer_B3_3c, answer_B3_3d, answer_B3_3e, answer_B3_3f, answer_B3_3g, answer_B3_3h, answer_B3_3i]):
    wb.check(f"B3.3{letter}", answer)

**Ex B3.4 — Encodage sinusoïdal : quelques valeurs et l'effet d'un décalage de position**

In [ ]:
# Ex B3.4: the values you found on paper
answer_B3_4a = ...  # a) a whole number
answer_B3_4b = ...  # b) a number, 3 decimals
answer_B3_4c = ...  # c) a number, 4 decimals
answer_B3_4d = ...  # d) a number, 4 decimals
answer_B3_4e = ...  # e) a number, 4 decimals
answer_B3_4f = ...  # f) a number, 4 decimals
answer_B3_4g = ...  # g) a number, 4 decimals
answer_B3_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B3_4a, answer_B3_4b, answer_B3_4c, answer_B3_4d, answer_B3_4e, answer_B3_4f, answer_B3_4g, answer_B3_4h]):
    wb.check(f"B3.4{letter}", answer)

**Ex B3.5 — Compter les paramètres d'un GPT et retrouver les 124 millions de GPT-2 small**

In [ ]:
# Ex B3.5: the values you found on paper
answer_B3_5a = ...  # a) a whole number
answer_B3_5b = ...  # b) a whole number
answer_B3_5c = ...  # c) a whole number
answer_B3_5d = ...  # d) a whole number
answer_B3_5e = ...  # e) a whole number
answer_B3_5f = ...  # f) a whole number
answer_B3_5g = ...  # g) a whole number
answer_B3_5h = ...  # h) a number, 1 decimal
answer_B3_5i = ...  # i) True or False
answer_B3_5j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_B3_5a, answer_B3_5b, answer_B3_5c, answer_B3_5d, answer_B3_5e, answer_B3_5f, answer_B3_5g, answer_B3_5h, answer_B3_5i, answer_B3_5j]):
    wb.check(f"B3.5{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une attention à la main, masque causal compris, et dire pourquoi on divise par $\sqrt{d_k}$ ?
2. Sais-tu suivre les formes d'une attention multi-têtes et compter les paramètres d'un GPT ?
3. Sais-tu dire ce que font un encodage sinusoïdal et RoPE, et démontrer la propriété de RoPE ?
4. Sais-tu chiffrer le calcul d'un entraînement et la mémoire d'un KV cache ?

**Pour aller plus loin** : la vidéo « Let's build GPT » d'A. Karpathy et *The Annotated Transformer*, cités dans la fiche. La suite : le bonus B4, les LLM en pratique.

*Les exercices de code B3.12 à B3.30 (partie A : ta librairie NumPy ; partie B : le mini-GPT en PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B3` pour obtenir le notebook complet.*